# How does a Bike-share navigate speedy success?

## Overview

This notebook demonstrates end-to-end data analysis workflow focusing on:

- Data preparation and quality assessment
- Exploratory analysis and pattern discovery
- Visualization of key insights
- Actionable recommendations

**Key question:** What distinguishes casual riders from annual members, and how can we convert more casual riders into memberships?

**Data source:** [Public Divvy trip data](https://divvy-tripdata.s3.amazonaws.com/index.html)

**Privacy and reproducibility:**
- This aggregated analysis uses only trip-level metadata with no personal/individual identifiers
- Raw data files are expected under `data/raw/` (this directory is git-ignored due to file size). Download the datasets from the source URL above and store each file using the following structure: `data/raw/{YYYYMM}-divvy-tripdata/{YYYYMM}-divvy-tripdata.csv`, where `YYYYMM` represents the year and month (e.g., `202601`)
- All transformations are documented in this notebook
- Processed data outputs are saved to `data/processed/` for downstream use

In [1]:
"""Demonstrates a complete data analysis pipeline."""

import pandas as pd
import numpy as np
from pathlib import Path

## Load data

Load up to the most recent 12 months of trip data, stored as individual CSV files in `data/raw/`

In [2]:
# Get list of raw data files sorted by date (up to 12 most recent months)
base = Path('../data/raw')
files = sorted(base.glob("*-divvy-tripdata/*.csv"))[-12:]

# Load up to 12 most recent months
files_to_load = files[:12]
dfs = [pd.read_csv(file) for file in files]

# Extract months from folder names
months_pretty = [
    pd.to_datetime(f.parent.name[:6], format="%Y%m").strftime("%B %Y")
    for f in files_to_load
]

print(f"Loaded {len(dfs)} month(s) data files: {months_pretty}")

Loaded 12 month(s) data files: ['April 2025', 'May 2025', 'June 2025', 'July 2025', 'August 2025', 'September 2025', 'October 2025', 'November 2025', 'December 2025', 'January 2026', 'February 2026', 'March 2026']


## Standardize schema

Identify and correct columns not shared between datasets.

In [ ]:
# Fix most common inconsistencies: column names, types, and member/casual labels
updated_dfs = []
for df in dfs:
    df = df.rename(columns={
        'trip_id': 'ride_id', 'from_station_id': 'start_station_id',
        'from_station_name': 'start_station_name', 'start_time': 'started_at',
        'end_time': 'ended_at', 'to_station_id': 'end_station_id',
        'to_station_name': 'end_station_name', 'usertype': 'member_casual'
    })
    df['ride_id'] = df['ride_id'].astype(str)
    df['member_casual'] = df['member_casual'].replace({
        'Subscriber': 'member', 'Customer': 'casual'
    })
    updated_dfs.append(df)
dfs = updated_dfs

# List all unique columns
all_columns = set()
for df in dfs:
    all_columns.update(df.columns)

# Find shared columns
shared_columns = set(dfs[0].columns)
for df in dfs[1:]:
    shared_columns &= set(df.columns)
print("Shared columns:", shared_columns)

# Find missing columns in each file
unshared_columns = all_columns - shared_columns
if unshared_columns:
    for i, df in enumerate(dfs):
        missing = all_columns - set(df.columns)
        if missing:
            print(f"File {i} missing columns: {missing}")

Shared columns: {'member_casual', 'started_at', 'start_lat', 'start_lng', 'end_station_name', 'end_lng', 'ride_id', 'ended_at', 'rideable_type', 'start_station_id', 'end_station_id', 'start_station_name', 'end_lat'}


## Combine datasets

Bind rows across multiple dataframes and remove exact duplicates. Then inspect overall shape.

In [6]:
shared_columns = list(shared_columns)
all_trips = pd.concat([df[shared_columns] for df in dfs], ignore_index=True)
count_before_dedupe = len(all_trips)
all_trips = all_trips.drop_duplicates()
print(f"Deduplicated {count_before_dedupe - len(all_trips):,} rows")
print(f"Total trips: {len(all_trips):,}")

Deduplicated 0 rows
Total trips: 5,620,544


## Display summary statistics

Display summary statistics like row count, columns, data types, and missing values.

In [14]:
print("=== Dataset Overview ===")
print(f"Shape: {all_trips.shape[0]:,} rows x {all_trips.shape[1]} columns")

print("\n=== Data Types ===")
print(all_trips.dtypes)

print("\n=== Missing Values ===")
missing = all_trips.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No missing values detected")

=== Dataset Overview ===
Shape: 5,620,544 rows x 13 columns

=== Data Types ===
member_casual             str
started_at                str
start_lat             float64
start_lng             float64
end_station_name          str
end_lng               float64
ride_id                   str
ended_at                  str
rideable_type             str
start_station_id          str
end_station_id            str
start_station_name        str
end_lat               float64
dtype: object

=== Missing Values ===
end_station_name      1259214
end_lng                  5784
start_station_id      1194952
end_station_id        1259214
start_station_name    1194952
end_lat                  5784
dtype: int64


In [17]:
print("\n=== Basic Statistics ===")
print(all_trips.describe(include='all'))


=== Basic Statistics ===
       member_casual               started_at     start_lat     start_lng  \
count        5620544                  5620544  5.620544e+06  5.620544e+06   
unique             2                  5619472           NaN           NaN   
top           member  2025-04-23 18:08:46.683           NaN           NaN   
freq         3605045                        2           NaN           NaN   
mean             NaN                      NaN  4.190390e+01 -8.764657e+01   
std              NaN                      NaN  4.441489e-02  2.721675e-02   
min              NaN                      NaN  4.164850e+01 -8.789000e+01   
25%              NaN                      NaN  4.188241e+01 -8.766000e+01   
50%              NaN                      NaN  4.189897e+01 -8.764182e+01   
75%              NaN                      NaN  4.193000e+01 -8.763000e+01   
max              NaN                      NaN  4.207000e+01 -8.752000e+01   

       end_station_name       end_lng           r